## 03 — Baseline: SVM + Strict LOSO Evaluation

 EEG-based Alzheimer's Disease Classification


In [1]:
# %%
# Install dependencies

import subprocess, sys

def pip(*p):
    subprocess.run([sys.executable,'-m','pip','install',*p,'-q'], check=True)

pip('mne','awscli','scipy','numpy','pandas','scikit-learn','matplotlib','seaborn','tqdm','joblib')
print('Dependencies ready.')


Dependencies ready.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# %%
# Environment + paths

import os, pickle
from pathlib import Path

if os.path.exists('/content'):
    ENV, BASE = 'colab', Path('/content/sir-eegnet')
elif os.path.exists('/kaggle/working'):
    ENV, BASE = 'kaggle', Path('/kaggle/working/sir-eegnet')
else:
    ENV, BASE = 'local', Path.cwd().parent

DATA_DIR     = BASE / 'data'
DS_DIR       = DATA_DIR / 'ds004504'
DERIV_DIR    = DS_DIR / 'derivatives'
PARTICIPANTS = DS_DIR / 'participants.tsv'
FEATURES_DIR = BASE / 'features'
RESULTS_DIR  = BASE / 'results'

for d in [DATA_DIR, DS_DIR, DERIV_DIR, FEATURES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Env: {ENV}')


Env: local


In [3]:
# %%
# Constants

import numpy as np
import pandas as pd
np.random.seed(42)

SFREQ          = 500.0
EPOCH_DURATION = 4.0
EPOCH_OVERLAP  = 0.5

GROUP_MAP      = {'A': 0, 'F': 1, 'C': 2}
LABEL_NAMES    = {0: 'AD', 1: 'FTD', 2: 'HC'}

FREQ_BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
    'gamma': (30.0, 45.0),
}

print('Constants loaded.')


Constants loaded.


In [4]:
# %%
# Feature store loading / building

STORE_PATH = FEATURES_DIR / 'subject_data_ds004504.pkl'

def download_ds004504(deriv_dir, participants_path):
    if participants_path.exists() and len(list(deriv_dir.rglob('*.set'))) >= 80:
        return
    print('Downloading ds004504...')
    r1 = subprocess.run(['aws','s3','sync',
        's3://openneuro.org/ds004504/derivatives/', str(deriv_dir),
        '--no-sign-request','--region','us-east-1'], capture_output=True, text=True)
    if r1.returncode != 0: raise RuntimeError(r1.stderr[-600:])
    r2 = subprocess.run(['aws','s3','cp',
        's3://openneuro.org/ds004504/participants.tsv', str(participants_path),
        '--no-sign-request','--region','us-east-1'], capture_output=True, text=True)
    if r2.returncode != 0: raise RuntimeError(r2.stderr[-600:])
    print('Download complete.')


In [5]:
# %%
# Feature builder (fallback if cache missing)

def build_features_from_scratch():
    import mne
    from scipy.signal import welch
    mne.set_log_level('WARNING')

    def epoch_raw_signal(data, sfreq=500.0, duration=4.0, overlap=0.5):
        ep_s = int(duration*sfreq)
        step_s = int(ep_s*(1-overlap))
        return np.stack([data[:,s:s+ep_s] for s in range(0,data.shape[1]-ep_s+1,step_s)],axis=0).astype(np.float32)

    def compute_rbp(epoch, sfreq, freq_bands):
        nperseg = min(256, epoch.shape[1])
        freqs, psd = welch(epoch, fs=sfreq, nperseg=nperseg, axis=-1)
        res = freqs[1]-freqs[0]
        abs_p = np.stack([
            np.sum(psd[:,(freqs>=lo)&(freqs<=hi)],axis=-1)*res
            for _,(lo,hi) in freq_bands.items()
        ], axis=0)
        rbp = abs_p / (abs_p.sum(axis=0,keepdims=True)+1e-10)
        return rbp.T.flatten().astype(np.float32)

    download_ds004504(DERIV_DIR, PARTICIPANTS)

    parts = pd.read_csv(PARTICIPANTS, sep='\t')
    parts['label'] = parts['Group'].map(GROUP_MAP)
    parts = parts.dropna(subset=['label'])
    parts['label'] = parts['label'].astype(int)

    subject_data = {}

    for _, row in parts.iterrows():
        sid = row['participant_id']
        label = int(row['label'])
        fpath = DERIV_DIR / sid / 'eeg' / f'{sid}_task-eyesclosed_eeg.set'

        if not fpath.exists():
            continue

        try:
            raw = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=False)
            data = raw.get_data()
            epochs = epoch_raw_signal(data)

            rbp_ep = np.stack([compute_rbp(ep, SFREQ, FREQ_BANDS) for ep in epochs])

            subject_data[sid] = {
                'epochs_raw': epochs,
                'rbp': rbp_ep,
                'label': label,
                'group': row['Group'],
                'mmse': float(row.get('MMSE',np.nan)),
                'age': float(row['Age']),
                'n_epochs': len(epochs)
            }

        except Exception as e:
            print(f'  [WARN] {sid}: {e}')

    with open(STORE_PATH, 'wb') as f:
        pickle.dump(subject_data, f, protocol=4)

    print(f'Feature store built and saved: {len(subject_data)} subjects')
    return subject_data


In [6]:
# %%
# Load or build

if STORE_PATH.exists():
    print('Loading cached feature store...')
    with open(STORE_PATH, 'rb') as f:
        subject_data = pickle.load(f)
    print(f'Loaded {len(subject_data)} subjects from cache.')
else:
    print('Feature store not found — building from scratch...')
    subject_data = build_features_from_scratch()


Feature store not found — building from scratch...
Download complete.


/tmp/ipykernel_658/1229535410.py:43: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=False)
/tmp/ipykernel_658/1229535410.py:43: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=False)
/tmp/ipykernel_658/1229535410.py:43: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=False)
/tmp/ipykernel_658/1229535410.py:43: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(str(fpath), preload=True,

Feature store built and saved: 88 subjects


In [7]:
# %%
# Precompute LOSO folds (SAFE caching — no leakage)

print('Precomputing LOSO folds (this runs once)...')

all_subject_ids = list(subject_data.keys())

precomputed_folds = []

for test_sid in all_subject_ids:
    train_sids = [s for s in all_subject_ids if s != test_sid]

    # Build TRAIN
    X_train = np.concatenate(
        [subject_data[s]['rbp'] for s in train_sids],
        axis=0
    )
    y_train = np.concatenate(
        [np.array([subject_data[s]['label']] * subject_data[s]['n_epochs'])
         for s in train_sids]
    )

    # Build TEST
    X_test = subject_data[test_sid]['rbp']
    y_test = subject_data[test_sid]['label']

    precomputed_folds.append((X_train, y_train, X_test, y_test, test_sid))

print(f'Precomputed {len(precomputed_folds)} folds.')


Precomputing LOSO folds (this runs once)...
Precomputed 88 folds.


In [8]:
# %%
# LOSO evaluation (using cached folds)

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def majority_vote(epoch_preds):
    return Counter(epoch_preds).most_common(1)[0][0]


def _process_fold(fold_data, clf_factory):
    X_train, y_train, X_test, y_test, test_sid = fold_data
    
    # Scale INSIDE fold (correct)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Train
    clf = clf_factory()
    clf.fit(X_train_scaled, y_train)

    # Predict
    epoch_preds = clf.predict(X_test_scaled)
    subject_pred = majority_vote(epoch_preds)

    res = {
        'true': y_test,
        'pred': subject_pred,
    }
    
    if hasattr(clf, 'predict_proba'):
        res['pred_proba'] = clf.predict_proba(X_test_scaled).mean(axis=0)
        
    return res


def loso_evaluate_svm_cached(precomputed_folds, clf_factory, label_names, n_jobs=-1, verbose=True):
    print(f"Running LOSO with {len(precomputed_folds)} folds in parallel (n_jobs={n_jobs})... ")
    
    results = Parallel(n_jobs=n_jobs)(
        delayed(_process_fold)(fold, clf_factory) 
        for fold in tqdm(precomputed_folds, disable=not verbose)
    )

    true_labels = [r['true'] for r in results]
    pred_labels = [r['pred'] for r in results]
    pred_probas = [r.get('pred_proba') for r in results]
    pred_probas = [p for p in pred_probas if p is not None]

    return {
        'true': np.array(true_labels),
        'pred': np.array(pred_labels),
        'accuracy': accuracy_score(true_labels, pred_labels),
        'f1_weighted': f1_score(true_labels, pred_labels, average='weighted', zero_division=0),
        'f1_macro': f1_score(true_labels, pred_labels, average='macro', zero_division=0),
        'pred_proba': np.array(pred_probas) if pred_probas else None
    }

print('Cached LOSO engine ready.')


Cached LOSO engine ready.


In [ ]:
from sklearn.svm import SVC

def svm_factory_3class():
    return SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=False,
    cache_size=1000,   # caching new added
    random_state=42,
    class_weight='balanced'
)

results_svm_3 = loso_evaluate_svm_cached(
    precomputed_folds,
    svm_factory_3class,
    LABEL_NAMES
)

print(f'Accuracy: {results_svm_3["accuracy"]*100:.2f}%')


Running LOSO with 88 folds in parallel (n_jobs=-1)... 


  0%|          | 0/88 [00:00<?, ?it/s]

/opt/venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
# %%
# Precompute binary folds (AD vs HC)

binary_map = {0:0, 2:1}
binary_subjects = {}

for sid, d in subject_data.items():
    if d['label'] in binary_map:
        new_d = dict(d)
        new_d['label'] = binary_map[d['label']]
        binary_subjects[sid] = new_d

print('Precomputing binary folds...')

binary_ids = list(binary_subjects.keys())
binary_folds = []

for test_sid in binary_ids:
    train_sids = [s for s in binary_ids if s != test_sid]

    X_train = np.concatenate(
        [binary_subjects[s]['rbp'] for s in train_sids],
        axis=0
    )
    y_train = np.concatenate(
        [np.array([binary_subjects[s]['label']] * binary_subjects[s]['n_epochs'])
         for s in train_sids]
    )

    X_test = binary_subjects[test_sid]['rbp']
    y_test = binary_subjects[test_sid]['label']

    binary_folds.append((X_train, y_train, X_test, y_test, test_sid))

print(f'Binary folds ready: {len(binary_folds)}')


In [ ]:
# %%
def svm_factory_binary():
    return SVC(kernel='rbf', C=1.0, gamma='scale',
               probability=False, random_state=42, class_weight='balanced')

results_svm_bin = loso_evaluate_svm_cached(
    binary_folds,
    svm_factory_binary,
    {0:'AD',1:'HC'}
)

print(f'Binary Accuracy: {results_svm_bin["accuracy"]*100:.2f}%')


In [ ]:
# %%
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
sns.set_style('white')
plt.rcParams.update({'font.size':12})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3-class
ax = axes[0]
cm3 = confusion_matrix(results_svm_3['true'], results_svm_3['pred'], labels=[0,1,2])
sns.heatmap(cm3, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['AD','FTD','HC'], yticklabels=['AD','FTD','HC'],
            cbar=False, linewidths=0.5)
ax.set_title(f'SVM 3-class LOSO\nAcc={results_svm_3["accuracy"]*100:.1f}%  F1={results_svm_3["f1_weighted"]*100:.1f}%',
             fontweight='bold')
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')

# Binary
ax2 = axes[1]
cm2 = confusion_matrix(results_svm_bin['true'], results_svm_bin['pred'], labels=[0,1])
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', ax=ax2,
            xticklabels=['AD','HC'], yticklabels=['AD','HC'],
            cbar=False, linewidths=0.5)
ax2.set_title(f'SVM Binary (AD vs HC) LOSO\nAcc={results_svm_bin["accuracy"]*100:.1f}%  F1={results_svm_bin["f1_weighted"]*100:.1f}%',
              fontweight='bold')
ax2.set_ylabel('True Label'); ax2.set_xlabel('Predicted Label')

plt.tight_layout()
fig.savefig(str(RESULTS_DIR/'fig04_svm_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig04_svm_confusion_matrices.png')


# %%
from sklearn.model_selection import StratifiedKFold

print('=== Leakage Demonstration ===')
print('Comparing epoch-level CV (wrong) vs subject-level LOSO (correct) for AD vs HC')
print()

# Build flat arrays (ALL subjects together — the WRONG way)
X_all, y_all = [], []
for sid, d in binary_subjects.items():
    X_all.append(d['rbp'])
    y_all.extend([d['label']] * d['n_epochs'])
X_all = np.concatenate(X_all, axis=0)
y_all = np.array(y_all)

# Epoch-level 5-fold CV (leaky — wrong evaluation)
scaler_leak = StandardScaler().fit(X_all)
X_all_scaled = scaler_leak.transform(X_all)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
leaky_accs = []

for tr_idx, te_idx in skf.split(X_all_scaled, y_all):
    clf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, class_weight='balanced')
    clf.fit(X_all_scaled[tr_idx], y_all[tr_idx])
    leaky_accs.append(accuracy_score(y_all[te_idx], clf.predict(X_all_scaled[te_idx])))

leaky_acc  = np.mean(leaky_accs)
correct_acc = results_svm_bin['accuracy']

print(f'Epoch-level 5-fold CV (LEAKY)     : {leaky_acc*100:.2f}%  ← inflated, NOT real')
print(f'Subject-level LOSO (CORRECT)      : {correct_acc*100:.2f}%  ← honest, real-world estimate')
print(f'Inflation due to leakage          : {(leaky_acc - correct_acc)*100:.2f} percentage points')
print()
print('This gap explains why literature reports 99% while real performance is much lower.')
print('Our paper uses the CORRECT evaluation throughout.')


# %%
import json

svm_results_summary = {
    'model': 'SVM (RBF kernel)',
    'evaluation': 'Subject-level LOSO + majority vote',
    'multiclass_AD_FTD_HC': {
        'accuracy': round(float(results_svm_3['accuracy']), 4),
        'f1_weighted': round(float(results_svm_3['f1_weighted']), 4),
        'f1_macro': round(float(results_svm_3['f1_macro']), 4),
    },
    'binary_AD_HC': {
        'accuracy': round(float(results_svm_bin['accuracy']), 4),
        'f1_weighted': round(float(results_svm_bin['f1_weighted']), 4),
        'f1_macro': round(float(results_svm_bin['f1_macro']), 4),
    },
    'leakage_demo': {
        'epoch_level_leaky': round(float(leaky_acc), 4),
        'subject_level_correct': round(float(correct_acc), 4),
        'inflation_pp': round(float((leaky_acc-correct_acc)*100), 2),
    }
}

with open(str(RESULTS_DIR / 'results_svm_baseline.json'), 'w') as f:
    json.dump(svm_results_summary, f, indent=2)

# Save per-subject predictions
pred_df = pd.DataFrame({
    'subject_id': list(binary_subjects.keys()),
    'true_label': results_svm_bin['true'],
    'pred_label': results_svm_bin['pred'],
    'correct': (results_svm_bin['true'] == results_svm_bin['pred']).astype(int)
})

pred_df.to_csv(str(RESULTS_DIR/'svm_subject_predictions.csv'), index=False)

print('=== Notebook 03 Complete ===')
print()

for k, v in svm_results_summary.items():
    if isinstance(v, dict):
        print(f'{k}:')
        for kk, vv in v.items():
            print(f'  {kk}: {vv}')

print()
print('Files saved:')
print(f'  {RESULTS_DIR}/results_svm_baseline.json')
print(f'  {RESULTS_DIR}/svm_subject_predictions.csv')
print()
print('Next: run 04_baseline_eegnet_loso.ipynb')
